# Document RAG System using LangChain, Google Gemini & ChromaDB

This notebook implements the complete Retrieval-Augmented Generation (RAG) pipeline using **LangChain**, **Google Generative AI (Gemini)**, and **ChromaDB**.

### Pipeline Steps:
1. **Install Dependencies** (`langchain`, `langchain-community`, `langchain-chroma`, `langchain-google-genai`, `chromadb`, `pypdf`)
2. **API Key Setup** (Configuring Gemini API key)
3. **PDF Upload** (Interactive upload via Google Colab)
4. **Document Loading** (`PyPDFLoader`)
5. **Text Splitting / Chunking** (`RecursiveCharacterTextSplitter`)
6. **Embeddings & Vector Store** (`GoogleGenerativeAIEmbeddings` & `Chroma`)
7. **Retriever & LLM Configuration** (`ChatGoogleGenerativeAI`)
8. **LCEL RAG Chain** (LangChain Expression Language pipeline)
9. **Interactive Query Execution**

### Step 1: Install Dependencies
Install LangChain core, integration packages for Google GenAI & Chroma, and PDF extraction tools.

In [1]:
# Install LangChain and Google GenAI packages
!pip -q install langchain langchain-community langchain-core langchain-chroma langchain-google-genai pypdf chromadb

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


### Step 2: Configure Gemini API Key
Retrieve the Gemini API key from Colab Secrets or prompt for it.

In [2]:
import os
import getpass

# Check for Colab Secrets first, then fallback to user input
try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    api_key = None

if not api_key:
    api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")

if not api_key:
    api_key = getpass.getpass("Enter your Google Gemini API Key: ")

os.environ["GOOGLE_API_KEY"] = api_key
print("Gemini API key successfully configured for LangChain!")

Gemini API key successfully configured for LangChain!


### Step 3: PDF File Upload
Upload the PDF document (such as the iPhone 16 Pro Environmental Report) to your environment.

In [4]:
import os
from google.colab import files

# Check if there are already PDF files in the current directory
pdf_files = [f for f in os.listdir('.') if f.lower().endswith('.pdf')]

if pdf_files:
    print(f"Found existing PDF file(s) in workspace: {pdf_files}")
    use_existing = input("Do you want to use the existing file? (y/n): ").strip().lower()
    if use_existing == 'y':
        pdf_path = pdf_files[0]
        print(f"Using existing file: {pdf_path}")
    else:
        print("Please select a new PDF file to upload:")
        uploaded = files.upload()
        pdf_path = list(uploaded.keys())[0]
        print(f"\nFile ready for parsing: {pdf_path}")
else:
    print("No existing PDF found. Please select your PDF file to upload:")
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]
    print(f"\nFile ready for parsing: {pdf_path}")

No existing PDF found. Please select your PDF file to upload:


Saving iPhone_16_Pro_and_iPhone_16_Pro_Max_PER_Sept2024.pdf to iPhone_16_Pro_and_iPhone_16_Pro_Max_PER_Sept2024.pdf

File ready for parsing: iPhone_16_Pro_and_iPhone_16_Pro_Max_PER_Sept2024.pdf


### Step 4: Extract and Load PDF Document
We use LangChain's `PyPDFLoader` to parse the document pages and their metadata.

In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_path)
docs = loader.load()

total_chars = sum(len(doc.page_content) for doc in docs)
print(f"Loaded {len(docs)} pages with a total of {total_chars} characters from '{pdf_path}'.")

/tmp/ipykernel_2030/2993431524.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 12 pages with a total of 27627 characters from 'iPhone_16_Pro_and_iPhone_16_Pro_Max_PER_Sept2024.pdf'.


### Step 5: Text Chunking
We use LangChain's `RecursiveCharacterTextSplitter` to break the document into chunks while preserving paragraph boundaries and semantic continuity.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Chunk size 1000 chars, overlap 200 chars (consistent with original pipeline)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

document_chunks = text_splitter.split_documents(docs)
print(f"Split the document into {len(document_chunks)} discrete chunks.")

Split the document into 39 discrete chunks.


### Step 6: Initialize Embeddings and Chroma Vector Store
We initialize LangChain's `GoogleGenerativeAIEmbeddings` and store chunk embeddings in a local `Chroma` collection in batches to stay within free-tier rate limits.

In [14]:
import time
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    task_type="retrieval_document"
)

# Create local ephemeral Chroma vector store
vectorstore = Chroma(
    collection_name="Apple16_langchain",
    embedding_function=embeddings
)

# Insert chunks in throttled batches to respect Gemini rate limits (15 RPM free tier)
batch_size = 5
print(f"Embedding and inserting {len(document_chunks)} chunks in throttled batches...")

for i in range(0, len(document_chunks), batch_size):
    batch = document_chunks[i:i + batch_size]
    vectorstore.add_documents(batch)
    print(f"  - Successfully uploaded chunks {i} to {min(i + batch_size, len(document_chunks))}")

    if i + batch_size < len(document_chunks):
        time.sleep(5)

print("\nAll chunks successfully embedded and stored in Chroma vector store!")

Embedding and inserting 39 chunks in throttled batches...
  - Successfully uploaded chunks 0 to 5
  - Successfully uploaded chunks 5 to 10
  - Successfully uploaded chunks 10 to 15
  - Successfully uploaded chunks 15 to 20
  - Successfully uploaded chunks 20 to 25
  - Successfully uploaded chunks 25 to 30
  - Successfully uploaded chunks 30 to 35
  - Successfully uploaded chunks 35 to 39

All chunks successfully embedded and stored in Chroma vector store!


### Step 7: Create Retriever and LLM
Create a retriever to pull top-3 matching chunks and initialize the Gemini Chat model.

In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Configure retriever for top 3 results
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Initialize Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.0
)
print("Retriever and Gemini LLM successfully initialized!")

Retriever and Gemini LLM successfully initialized!


### Step 8: Build the RAG Chain using LCEL (LangChain Expression Language)
Define the system prompt and compose the pipeline with `RunnablePassthrough` and `StrOutputParser`.

In [19]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Prompt template matching original instructions
template = """You are a precise document assistant. Use only the provided document context below to answer the user's question accurately. If you do not know the answer or if it is not explicitly mentioned in the context, say that you cannot find it.

Context:
{context}

Question: {question}"""

prompt = ChatPromptTemplate.from_template(template)

# Helper function to format retrieved documents
def format_docs(docs):
    return "\n---\n".join(doc.page_content for doc in docs)

# Composed RAG Chain using modern LangChain syntax (LCEL)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG Chain successfully constructed!")

RAG Chain successfully constructed!


### Step 9: Interactive Query Execution
Type your query below and run the cell. We retrieve the matching document chunks and stream/generate the final answer.

In [20]:
#@markdown ### Enter Your Document Query:
user_query = "How much of the packaging is fiber-based?" #@param {type:"string"}

if not user_query.strip():
    print("Please enter a query to run the pipeline!")
else:
    print(f"Processing Query: {user_query}\n")

    # 1. Retrieve the matching chunks
    retrieved_docs = retriever.invoke(user_query)

    # 2. Display context chunks retrieved
    print("=== RETRIEVED CONTEXT SAMPLES ===")
    for idx, doc in enumerate(retrieved_docs, 1):
        content_preview = doc.page_content[:150].replace('\n', ' ').strip()
        page_num = doc.metadata.get('page', 'Unknown')
        print(f"Chunk {idx} (Page {page_num}): {content_preview}...")
    print("=================================\n")

    # 3. Generate answer through the LangChain pipeline
    print("=== GEMINI GENERATED ANSWER ===")
    answer = rag_chain.invoke(user_query)
    print(answer)

Processing Query: How much of the packaging is fiber-based?

=== RETRIEVED CONTEXT SAMPLES ===
Chunk 1 (Page 5): iPhone 16 Pro and iPhone 16 Pro Max  |  Product Environmental Report 6 100%  of the packaging18   is fiber-based, due to  our work to remove  plastic...
Chunk 2 (Page 2): Use   Minimize emissions from  product use, enhance product  longevity, and seek to avoid  exposure to harmful chemicals Make  Uphold high standards o...
Chunk 3 (Page 5): from responsibly managed forests. 19 And we have protected or created enough responsibly  managed forests to cover all the virgin wood fiber we use in...

=== GEMINI GENERATED ANSWER ===


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided document, 100% of the packaging is fiber-based.
